# 11 하이브리드 수요예측 — SBC vs ML scheme 비교

**2-type(고변동 E · 저변동 C)** 에 대해, 10장과 동일한 **XGBoost + Best 임베딩** 예측으로
**SBC(rule-base) vs ML(AE+KMeans) 클러스터링 scheme**을 **제품수 가중 WMAPE**로 비교합니다.

## 논문(§4.5·5.3)과의 대응
| 논문 (Daiso 2센터) | 본 실습 (Ecuador 2-type) |
|---|---|
| Center A (저변동) → **ML(PatchTST-GMM)** 우세 | type **C** (저변동, System CV 최저) |
| Center B (고변동) → **SBC(rule-base)** 우세 | type **E** (고변동, System CV 최고) |
| WMAPE = Σ n_k·MAPE_k / Σ n_k (**제품수 가중**) | 동일 정의 |
| 가설: 저변동→ML, 고변동→SBC | 본 실습에서 검증 |

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 792 | 조건: 12


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import validation_weights, build_family_from_phase2_best, thesis_wmape_by_type
from utils.stats_summary import type_variation_table
from utils.config import selected_type_list

# 조건별 XGBoost + Best 임베딩(10장) 제품 단위 결과
val_weights = validation_weights(df)
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

# 논문식 WMAPE = Σ n_k·MAPE_k / Σ n_k (클러스터 제품수 가중, §4.5 Eq.50)
type_wmape = thesis_wmape_by_type(family_final)
print('=== type별 SBC vs ML (제품수 가중 WMAPE) ===')
display(type_wmape)
print('scheme 우세:', type_wmape['better_scheme'].value_counts().to_dict())

# System-Level CV(고/저변동 판별 지표)와 대조
var = type_variation_table(df)[['sys_cv', 'sku_mean_cv']].round(3)
sel = selected_type_list()  # [고변동, 저변동]
out = type_wmape.join(var)
out['variation'] = np.where(out.index == sel[0], 'high(고변동)',
                    np.where(out.index == sel[1], 'low(저변동)', ''))
print('=== scheme 우세 × System-Level CV ===')
display(out[['variation', 'sys_cv', 'SBC_wmape', 'ML_wmape', 'better_scheme']])

=== type별 SBC vs ML (제품수 가중 WMAPE) ===


,SBC_wmape,ML_wmape,delta_SBC_minus_ML,better_scheme
type,,,,
C,73.77,138.15,-64.38,SBC
E,39.63,74.63,-35.00,SBC


scheme 우세: {'SBC': 2}
=== scheme 우세 × System-Level CV ===


,variation,sys_cv,SBC_wmape,ML_wmape,better_scheme
type,,,,,
C,low(저변동),0.259,73.77,138.15,SBC
E,high(고변동),0.429,39.63,74.63,SBC


### ② 해석 — SBC vs ML scheme (제품수 가중 WMAPE)

**WMAPE = Σ_k n_k·MAPE_k / Σ_k n_k** (클러스터 제품수 가중, 논문 §4.5 Eq.50)

#### 결과 (2-type)
| type | System CV | SBC WMAPE | ML WMAPE | 우세 |
|------|-----------|-----------|----------|------|
| **C** (저변동) | 0.259 | **73.8** | 138.2 | **SBC** |
| **E** (고변동) | 0.429 | **39.6** | 74.6 | **SBC** |

→ 본 축소 실습에서는 **두 type 모두 SBC(rule-base) 우세**.

#### 왜 SBC가 이기는가 (축소 데이터의 구조적 이유)
- **ML 클러스터링이 [62, 4]로 퇴화** (06장): 66개 시계열이 구조적으로 유사해 임베딩 군집이 "메가셀러 4 + 롱테일 62"로만 나뉨. 롱테일 62개를 **한 패널**에 묶으면 이질적 규모·패턴이 섞여 예측이 나빠짐 (ML cluster2 WMAPE 급등).
- **SBC는 ADI·CV² 기반 4분류**로 Smooth/Intermittent/Erratic/Lumpy를 분리 → 각 조건 패널이 **동질적**이라 XGBoost가 패턴을 더 잘 학습.
- 즉 **동질성 기반 세분화(SBC) > 빈약한 임베딩 군집(ML)** 이 축소 데이터에서의 결과.

#### 논문과의 대응·차이
- 논문(12,661 SKU): ML 임베딩 군집이 **다중·풍부**해 저변동 센터 A에서 ML 우세, 고변동 센터 B에서 SBC 우세 (변동성↔scheme 연관).
- 본 실습(66 시계열): ML 군집 구조가 **빈약**(K=2, 한 군집 독식)해 scheme 우열이 **변동성과 단순 대응하지 않고 SBC가 일관 우세**.
- **결론:** SBC vs ML 우열은 **데이터 규모와 ML 군집 품질**에 크게 의존. 표본이 크고 변동 구조가 선명한 데이터(논문·Daiso)에서 ML의 강점이 드러나며, 축소 교육용 데이터에서는 SBC의 패턴 기반 분류가 안정적이다.

> **한계:** Ecuador 2-type·66 시계열·13주 test에 한정. 다른 규모·산업·클러스터 수에서 결과가 달라질 수 있음.